# YOLOv8 커스텀 학습: glass / cap / watch

1. **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. 왼쪽 🔑(보안 비밀)에 `ROBOFLOW_API_KEY` 추가 → 노트북 액세스 켜기 (키를 코드에 적지 않기 위함. GitHub 공개 저장소에 키가 올라가면 안 됨)
3. 위에서부터 순서대로 실행

In [ ]:
!pip install -q ultralytics roboflow lap
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 → 런타임 유형을 T4 GPU로 바꾸세요")

## 1. Roboflow에서 데이터셋 다운로드 (Version 1, 537장)

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
dataset = rf.workspace("yolomachinelearning").project("glass-cap-watch-detection").version(1).download(
    model_format="yolov8", location="/content/glass-cap-watch-v1")
print("다운로드 위치:", dataset.location)

## 2. 구간(block) 단위로 다시 분할

Roboflow의 무작위 70/20/10 분할은 연속 프레임(0.3초 간격)이 train과 valid에 섞여 **검증 점수가 부풀려짐(data leakage)**.
→ 영상별로 연속 프레임 `BLOCK`장(약 3초)을 한 묶음으로 만들어 묶음 단위로 train/valid/test에 배정.

In [ ]:
import re, random, shutil, yaml
from pathlib import Path
from collections import defaultdict, Counter

BLOCK = 10                      # 3.5fps 기준 약 3초
SRC = Path(dataset.location)
DST = Path("/content/data_block")
names = yaml.safe_load(open(SRC / "data.yaml"))["names"]

def make_yaml(root, out):
    yaml.safe_dump({"path": str(root), "train": "train/images", "val": "valid/images",
                    "test": "test/images", "names": names}, open(out, "w"), allow_unicode=True)
    return str(out)

# 파일명 예: glasscup_0059_jpg.rf.<hash>.jpg → (영상 glasscup, 프레임 59)
blocks = defaultdict(list)
for p in SRC.glob("*/images/*.jpg"):
    m = re.match(r"([a-z]+)_(\d+)_jpg", p.name)
    assert m, f"예상과 다른 파일명: {p.name}"
    blocks[(m[1], int(m[2]) // BLOCK)].append(p)

rng = random.Random(0)
split_of = {}
for video in sorted({v for v, _ in blocks}):
    keys = sorted(k for k in blocks if k[0] == video)
    rng.shuffle(keys)
    for i, k in enumerate(keys):
        split_of[k] = "train" if i < 0.7 * len(keys) else "valid" if i < 0.9 * len(keys) else "test"

shutil.rmtree(DST, ignore_errors=True)
count = Counter()
for k, imgs in blocks.items():
    s = split_of[k]
    for p in imgs:
        lbl = p.parent.parent / "labels" / (p.stem + ".txt")
        for f, sub in [(p, "images"), (lbl, "labels")]:
            (DST / s / sub).mkdir(parents=True, exist_ok=True)
            if f.exists():
                shutil.copy(f, DST / s / sub / f.name)
        count[s] += 1

BLOCK_YAML = make_yaml(DST, DST / "data.yaml")
RANDOM_YAML = make_yaml(SRC, SRC / "data_fixed.yaml")   # Roboflow 원래 분할 (비교용)
print("클래스:", names)
print("구간 분할 결과:", dict(count))

## 3. 학습 (구간 분할, yolov8n 사전학습 가중치 = 전이학습)

In [ ]:
from ultralytics import YOLO

EPOCHS = 50
model = YOLO("yolov8n.pt")
model.train(data=BLOCK_YAML, epochs=EPOCHS, imgsz=640, seed=0, name="block")
BLOCK_DIR = Path(model.trainer.save_dir)
BEST = BLOCK_DIR / "weights" / "best.pt"
print("학습 결과 폴더:", BLOCK_DIR)

In [ ]:
from IPython.display import Image, display
for f in ["results.png", "confusion_matrix.png", "BoxPR_curve.png"]:
    if (BLOCK_DIR / f).exists():
        print(f); display(Image(filename=str(BLOCK_DIR / f), width=800))

## 4. test 세트로 최종 성능 측정 (보고서용 표)

In [ ]:
m = YOLO(BEST).val(data=BLOCK_YAML, split="test", name="block_test")
print(f"{'class':<8}{'mAP50':>8}{'mAP50-95':>10}")
for i, c in enumerate(m.box.ap_class_index):
    print(f"{names[c]:<8}{m.box.ap50[i]:>8.3f}{m.box.ap[i]:>10.3f}")
print(f"{'all':<8}{m.box.map50:>8.3f}{m.box.map:>10.3f}")

## 5. 데모 영상 추적 + 이동 궤적

왼쪽 📁 파일 패널에 `demo_5s.mp4`를 업로드하거나 아래 셀에서 업로드.
`model.track()`으로 ID를 유지하고, ID별 중심점 기록을 선으로 그림(슬라이드 51: trajectory 저장은 코드 수정 필요).

In [ ]:
import os
if not os.path.exists("demo_5s.mp4"):
    from google.colab import files
    files.upload()   # demo_5s.mp4 선택

In [ ]:
import os, cv2, numpy as np

def track_video(weights, src, dst, conf=0.25):
    model = YOLO(weights)
    cap = cv2.VideoCapture(src)
    fps, w, h = cap.get(cv2.CAP_PROP_FPS), int(cap.get(3)), int(cap.get(4))
    out = cv2.VideoWriter("_raw.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    trail, hit, n = defaultdict(list), Counter(), 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        r = model.track(frame, persist=True, tracker="bytetrack.yaml", conf=conf, verbose=False)[0]
        n += 1
        hit.update({r.names[c] for c in r.boxes.cls.int().tolist()})
        img = r.plot(line_width=3)
        if r.boxes.id is not None:
            for (x, y, _, _), tid in zip(r.boxes.xywh.tolist(), r.boxes.id.int().tolist()):
                trail[tid] = (trail[tid] + [(int(x), int(y))])[-90:]     # 최근 90프레임(1.5초) 궤적
                cv2.polylines(img, [np.array(trail[tid], np.int32)], False, (0, 255, 255), 4)
        out.write(img)
    cap.release(); out.release()
    os.system(f"ffmpeg -loglevel error -y -i _raw.mp4 -c:v libx264 -pix_fmt yuv420p {dst} && rm _raw.mp4")  # 브라우저/GitHub 재생용
    print(f"{dst}: {n}프레임, 부여된 track ID {len(trail)}개 (3개에 가까울수록 추적 안정)")
    for k, v in hit.most_common():
        print(f"  {k:<12} 검출 프레임 {v}/{n} ({v/n:.0%})")
    return hit, n

track_video(BEST, "demo_5s.mp4", "demo_tracked.mp4")

In [ ]:
from IPython.display import HTML
from base64 import b64encode
HTML(f'<video width=480 controls src="data:video/mp4;base64,{b64encode(open("demo_tracked.mp4","rb").read()).decode()}">')

## 6. (선택) 학습 전후 비교 + 데이터 누수 비교 — 보고서 "경험한 내용"용

- 학습 전 기본 `yolov8n.pt`로 같은 데모 영상: glass를 cup/wine glass로 검출한 비율 (로컬 측정값 0%)
- Roboflow 무작위 분할로 학습했을 때 valid mAP가 구간 분할보다 얼마나 부풀려지는지

In [ ]:
track_video("yolov8n.pt", "demo_5s.mp4", "demo_baseline.mp4")

In [ ]:
rand = YOLO("yolov8n.pt")
rand.train(data=RANDOM_YAML, epochs=EPOCHS, imgsz=640, seed=0, name="random")
rv = YOLO(Path(rand.trainer.save_dir) / "weights" / "best.pt").val(data=RANDOM_YAML, name="random_val")
bv = YOLO(BEST).val(data=BLOCK_YAML, name="block_val")
print(f"valid mAP50  무작위 분할: {rv.box.map50:.3f}  |  구간 분할: {bv.box.map50:.3f}")
print("무작위 분할 점수가 더 높다면 → 이웃 프레임 누수로 부풀려진 것")

## 7. 결과 다운로드 (GitHub에 올릴 파일)

In [ ]:
shutil.copy(BEST, "best.pt")
!zip -qr results.zip best.pt demo_tracked.mp4 {BLOCK_DIR} /content/data_block/data.yaml
from google.colab import files
files.download("results.zip")